# VCARE CC Compass: staffing forecast and facility performance

How many **admin (MSA), LVN, and RN** hours each facility needs to clear its open review findings, how each facility's consult performance is trending, and how likely each facility is to meet its consult performance targets.

*All data is synthetic (600 fictitious consults, 6 hypothetical facilities). Workload minutes and targets are editable planning assumptions, not VA standards. Results are planning estimates, not staffing determinations.*

In [0]:
%sql
-- What date range and which category values are in the synthetic consults?
SELECT
  COUNT(*) AS consults,
  MIN(File_Entry_Date) AS first_entry, MAX(File_Entry_Date) AS last_entry,
  MIN(Authorized_Date) AS first_auth, MAX(Authorized_Date) AS last_auth,
  MAX(Last_Nurse_Action_Date) AS last_action,
  array_sort(collect_set(Consult_Status)) AS status_values,
  array_sort(collect_set(High_Risk)) AS high_risk_values,
  array_sort(collect_set(Expected_Max_Severity)) AS severity_values,
  array_sort(collect_set(Coordination_Level)) AS coordination_levels,
  array_sort(collect_set(Case_Type)) AS case_types,
  array_sort(collect_set(RFS_Decision)) AS rfs_decisions,
  COUNT_IF(Expected_Flag_Count > 0) AS cases_with_flags
FROM users.anna_fernandez.synthetic_cases

In [0]:
%sql
-- Each consult can have several expected flag codes. Split them so each flag type can be matched to a role.
SELECT trim(flag) AS flag_code, COUNT(*) AS consults,
       COUNT_IF(Consult_Status = 'Active') AS active_consults
FROM users.anna_fernandez.synthetic_cases
LATERAL VIEW explode(split(Expected_Flag_Codes, '; ')) f AS flag
WHERE Expected_Flag_Count > 0 AND trim(flag) <> ''
GROUP BY 1
ORDER BY consults DESC

In [0]:
# PLANNING ASSUMPTIONS: change these numbers, not the code below. They are not VA standards.
# Each expected flag code -> (role that resolves it, minutes of work to resolve one finding)
WORKLOAD = {
    "COORD_TIMELINESS":     ("RN",    30),  # coordination timeliness review
    "COORD_CANCELLED":      ("RN",    45),  # cancelled coordination follow-up
    "DEATH_REPORTED":       ("RN",    60),  # death reporting and documentation
    "RFS_CLOCK":            ("Admin", 20),  # RFS clock compliance
    "RFS_LETTER":           ("Admin", 15),  # RFS letter processing
    "RFS_SEQUENCE":         ("Admin", 15),  # RFS sequence check
    "NO_ACTION_LONG":       ("Admin", 20),  # long inactivity follow-up
    "BMI_CANCEL_REASON":    ("LVN",   15),  # BMI cancel reason documentation
    "BMI_EXCLUDED_SERVICE": ("LVN",   15),  # BMI excluded service review
    "BMI_NO_AGREEMENT":     ("LVN",   20),  # BMI agreement follow-up
    "BMI_SECOND_REVIEW":    ("LVN",   20),  # BMI second review documentation
}
NOT_A_FINDING = {"None", ""}

OPEN_STATUSES = ["Active"]      # open = Active only in the new schema
CLEAR_BACKLOG_WEEKS = 4         # goal: clear today's open findings within this many weeks
HOURS_PER_FTE_WEEK = 30         # productive hours per full-time employee per week (after meetings, leave, etc.)
RECENT_MONTHS = 3               # months of recent intake used for the ongoing-workload forecast
FORECAST_MONTHS = 3             # how far ahead to forecast

# Consult performance targets (share of consults meeting each criterion). Edit to match your local goals.
TARGETS = {
    "No_Flags":          0.65,  # share of consults with no expected flags
    "No_Critical":       0.90,  # share without Critical severity flags
    "No_High_Critical":  0.70,  # share without High or Critical severity
}
print(f"{len(WORKLOAD)} flag codes mapped to roles; targets: {TARGETS}")

In [0]:
# Turns every open finding (expected flag) into hours of work for the role that resolves it.
import pandas as pd, numpy as np

cases = spark.table("users.anna_fernandez.synthetic_cases").toPandas()
for c in ["File_Entry_Date", "Authorized_Date", "Care_Initiated_Date", "Last_Nurse_Action_Date"]:
    cases[c] = pd.to_datetime(cases[c])
cases["is_open"] = cases["Consult_Status"].isin(OPEN_STATUSES)
cases["entry_month"] = cases["File_Entry_Date"].dt.to_period("M")

# Explode Expected_Flag_Codes (semicolon+space delimited)
findings = cases.assign(flag_code=cases["Expected_Flag_Codes"].fillna("").str.split("; ")).explode("flag_code")
findings["flag_code"] = findings["flag_code"].str.strip()
unmapped = set(findings["flag_code"]) - set(WORKLOAD) - NOT_A_FINDING
assert not unmapped, f"Add these flag codes to WORKLOAD: {unmapped}"
findings = findings[findings["flag_code"].isin(WORKLOAD)].copy()
findings["role"] = findings["flag_code"].map(lambda f: WORKLOAD[f][0])
findings["hours"] = findings["flag_code"].map(lambda f: WORKLOAD[f][1]) / 60

ROLES = ["Admin", "LVN", "RN"]
open_f = findings[findings["is_open"]]
backlog_hours = open_f.pivot_table(index="Facility_Name", columns="role", values="hours", aggfunc="sum", fill_value=0).reindex(columns=ROLES, fill_value=0)
backlog_count = open_f.pivot_table(index="Facility_Name", columns="role", values="hours", aggfunc="count", fill_value=0).reindex(columns=ROLES, fill_value=0)
backlog_fte = backlog_hours / (CLEAR_BACKLOG_WEEKS * HOURS_PER_FTE_WEEK)

backlog = pd.concat({"open findings": backlog_count, "hours": backlog_hours.round(1),
                     f"FTE to clear in {CLEAR_BACKLOG_WEEKS} wks": backlog_fte.round(2)}, axis=1)
backlog.loc["VISN total"] = backlog.sum()
backlog = backlog.round(2)
backlog.columns = [f"{role} {measure}" for measure, role in backlog.columns]   # one-line column names
print(f"{len(open_f)} open findings across {cases['is_open'].sum()} active consults")
display(backlog.reset_index())

In [0]:
# New consults keep arriving and creating new findings. Forecast that ongoing work from recent intake,
# then add the backlog to get total staffing need per facility and role.
last_month = cases["entry_month"].max()
recent_months = pd.period_range(last_month - (RECENT_MONTHS - 1), last_month, freq="M")
recent_f = findings[findings["entry_month"].isin(recent_months)]
monthly_hours = (recent_f.pivot_table(index="Facility_Name", columns="role", values="hours", aggfunc="sum", fill_value=0)
                 .reindex(columns=ROLES, fill_value=0) / RECENT_MONTHS)
FTE_HOURS_PER_MONTH = HOURS_PER_FTE_WEEK * 52 / 12
ongoing_fte = monthly_hours / FTE_HOURS_PER_MONTH

# Month-by-month: backlog cleared in the first month(s), ongoing work every month
forecast_months = pd.period_range(pd.Period("2026-11", "M"), periods=FORECAST_MONTHS, freq="M")
backlog_months = CLEAR_BACKLOG_WEEKS * 12 / 52
rows = []
for i, m in enumerate(forecast_months):
    months_of_clearing = min(1.0, max(0.0, backlog_months - i))   # how much of this month is spent clearing backlog
    for fac in backlog_hours.index:
        for role in ROLES:
            hrs = monthly_hours.loc[fac, role] + backlog_hours.loc[fac, role] * months_of_clearing / backlog_months
            rows.append({"month": str(m), "facility": fac, "role": role, "hours": hrs, "FTE": hrs / FTE_HOURS_PER_MONTH})
monthly_plan = pd.DataFrame(rows)

staffing = pd.concat({"backlog FTE (first month)": backlog_fte.round(2), "ongoing FTE per month": ongoing_fte.round(2),
                      "peak FTE needed": (backlog_fte + ongoing_fte).round(2)}, axis=1)
staffing.loc["VISN total"] = staffing.sum()
staffing = staffing.round(2)
staffing.columns = [f"{role} {measure}" for measure, role in staffing.columns]
fmt = lambda p: p.strftime("%b %Y")
print(f"Data: consults entered {cases['File_Entry_Date'].min():%b %d, %Y} to {cases['File_Entry_Date'].max():%b %d, %Y}. "
      f"Ongoing workload is based on {fmt(recent_months[0])} to {fmt(recent_months[-1])} intake; "
      f"forecast covers {fmt(forecast_months[0])} to {fmt(forecast_months[-1])}.")
display(staffing.reset_index())
display(monthly_plan.pivot_table(index=["facility", "role"], columns="month", values="FTE").round(2).reset_index())

In [0]:
# Monthly consult performance per facility (by month the consult was entered), plus the direction of each trend.
import matplotlib.pyplot as plt

# Flag-based metrics replacing old compliance pillars
METRICS = {"No_Flags": "No expected flags", "No_Critical": "No critical severity",
           "No_High_Critical": "No high or critical"}
cases["No_Flags"] = np.where(cases["Expected_Flag_Count"] == 0, "Compliant", "Non-Compliant")
cases["No_Critical"] = np.where(cases["Expected_Max_Severity"] != "Critical", "Compliant", "Non-Compliant")
cases["No_High_Critical"] = np.where(~cases["Expected_Max_Severity"].isin(["High", "Critical"]), "Compliant", "Non-Compliant")
cases["days_to_auth"] = (cases["Authorized_Date"] - cases["File_Entry_Date"]).dt.days

flags = cases.assign(**{label: (cases[col] == "Compliant").astype(float) for col, label in METRICS.items()})
monthly = (flags.groupby(["Facility_Name", "entry_month"])
           .agg(consults=("Consult_Number", "size"), **{label: (label, "mean") for label in METRICS.values()},
                **{"Median days to authorization": ("days_to_auth", "median")})
           .reset_index())
monthly["month"] = monthly["entry_month"].dt.to_timestamp()

# Trend: straight-line slope over the months, in percentage points per month (positive = improving)
def slope(series):
    y = series.to_numpy(dtype=float); x = np.arange(len(y))
    ok = ~np.isnan(y)
    return np.polyfit(x[ok], y[ok], 1)[0] if ok.sum() >= 3 else np.nan

trend = monthly.sort_values("month").groupby("Facility_Name").agg(
    **{f"{label} (pts/month)": (label, lambda s: round(slope(s) * 100, 1)) for label in METRICS.values()},
    **{"Auth days (days/month)": ("Median days to authorization", lambda s: round(slope(s), 2))})
display(trend.reset_index())

# Charts: one panel per metric, one line per facility
VA_COLORS = ["#003F72", "#0071BC", "#FDB81E", "#5B616B", "#2E8540", "#981B1E"]
fig, axes = plt.subplots(1, 3, figsize=(18, 4.8), sharex=True)
for ax, label in zip(axes, ["No expected flags", "No critical severity", "Median days to authorization"]):
    for color, (fac, g) in zip(VA_COLORS, monthly.groupby("Facility_Name")):
        y = g[label] * (100 if label != "Median days to authorization" else 1)
        ax.plot(g["month"], y, marker="o", color=color, label=fac.replace(" VA Medical Center", "").replace(" VA Health System", ""))
    target = TARGETS.get({"No expected flags": "No_Flags", "No critical severity": "No_Critical"}.get(label, ""), None)
    if target is not None:
        ax.axhline(target * 100, color="#981B1E", linestyle="--", linewidth=1); ax.text(monthly["month"].min(), target * 100 + 1, "target", color="#981B1E")
    ax.set_title(label + (" (%)" if label != "Median days to authorization" else " (lower is better)"), color="#003F72", fontweight="bold")
    ax.grid(alpha=0.3); ax.tick_params(axis="x", rotation=45)
axes[0].legend(fontsize=8, loc="best")
fig.suptitle("Consult performance by facility and month of entry (synthetic data)", color="#003F72", fontsize=13)
plt.tight_layout(); plt.show()

In [0]:
# Chance each facility's TRUE flag rate meets the target, based on its most recent months.
# Method: a Beta-Binomial model (start from a flat prior, update with compliant vs. non-compliant consults).
# With about 12 synthetic consults per facility per month, the chances carry real uncertainty.
from scipy import stats

recent = cases[cases["entry_month"].isin(recent_months)]
rows = []
for fac, g in recent.groupby("Facility_Name"):
    for col, label in METRICS.items():
        n = len(g); x = int((g[col] == "Compliant").sum())
        target_val = TARGETS[col]
        posterior = stats.beta(1 + x, 1 + n - x)
        p_meet = 1 - posterior.cdf(target_val)
        lo, hi = posterior.ppf([0.05, 0.95])
        rows.append({"facility": fac, "metric": label, "target": f"{target_val:.0%}", "consults": n,
                     "current rate": f"{x / n:.0%}", "likely range (90%)": f"{lo:.0%} to {hi:.0%}",
                     "chance of meeting target": round(p_meet * 100, 1),
                     "gap to target (pts)": round((target_val - x / n) * 100, 1)})
prob = pd.DataFrame(rows)
prob["outlook"] = pd.cut(prob["chance of meeting target"], [-1, 10, 50, 90, 101],
                         labels=["Very unlikely", "Unlikely", "Likely", "Very likely"])
print(f"Based on consults entered {recent_months[0].strftime('%b')} to {recent_months[-1].strftime('%b %Y')}.")
display(prob.pivot_table(index="facility", columns="metric", values="chance of meeting target").round(1).reset_index())
display(prob.sort_values(["metric", "chance of meeting target"], ascending=[True, False]))

In [0]:
# Saves the three result tables to Unity Catalog so the dashboard can show them. Re-running replaces them.
SCHEMA = "users.anna_fernandez"

staffing_long = pd.concat([
    backlog_hours.stack().rename("backlog_hours"), backlog_count.stack().rename("open_findings"),
    backlog_fte.stack().rename("backlog_fte"), monthly_hours.stack().rename("ongoing_hours_per_month"),
    ongoing_fte.stack().rename("ongoing_fte")], axis=1).fillna(0).reset_index()
staffing_long.columns = ["facility", "role", "backlog_hours", "open_findings", "backlog_fte", "ongoing_hours_per_month", "ongoing_fte"]
staffing_long["peak_fte"] = staffing_long["backlog_fte"] + staffing_long["ongoing_fte"]
staffing_long["assumption_note"] = f"Clear backlog in {CLEAR_BACKLOG_WEEKS} weeks at {HOURS_PER_FTE_WEEK} productive hours per FTE week; minutes per finding are planning assumptions"

trends_out = monthly.drop(columns=["entry_month"]).rename(columns={"Facility_Name": "facility"})
trends_out.columns = [c.lower().replace(" ", "_") for c in trends_out.columns]

prob_out = prob.copy()
prob_out["outlook"] = prob_out["outlook"].astype(str)
prob_out.columns = [c.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("%", "pct") for c in prob_out.columns]

for name, df in {"cc_staffing_forecast": staffing_long, "cc_facility_monthly_trends": trends_out,
                 "cc_target_probability": prob_out}.items():
    spark.createDataFrame(df).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SCHEMA}.{name}")
    print(f"Saved {SCHEMA}.{name}: {spark.table(f'{SCHEMA}.{name}').count()} rows")

## What the numbers say (synthetic data, planning assumptions)

**1. Staffing to clear today's open findings** (138 findings on 366 active consults, cleared in 4 weeks)
* **Admin (MSA): about 23 hours (0.19 FTE for 4 weeks)**, for RFS clock/letter/sequence checks and long-inactivity follow-ups. Hawkeye Ridge needs the most admin time (5.2 hours).
* **RN: about 12 hours (0.09 FTE for 4 weeks)**, for coordination timeliness reviews. Every facility needs some RN time.
* **LVN: about 11 hours (0.10 FTE for 4 weeks)**, for BMI agreement, second review, and excluded-service documentation. Wakanda Shield needs the most LVN time (3.2 hours).
* After that, new consults add only about 0.05 RN and 0.01 admin FTE per month across the VISN. The synthetic set has only about 12 consults per facility per month, so real volumes scale these numbers up. The **role mix** is more balanced than before: about 51% admin, 24% LVN, and 25% RN hours.

**2. Trends (Mar to Sep 2026)**
* No-flags rate is improving fastest at Wakanda Shield (+9.8 pts/month) and Hawkeye Ridge (+3.4), but slipping at Iron Sentinel (-3.5) and Captain Liberty (-1.8).
* Days to authorization are slowly rising at Hawkeye Ridge (+0.25 days/month) and Iron Sentinel (+0.3).

**3. Chance of meeting targets** (Jul to Sep 2026 consults)
* **No expected flags (65%):** Hawkeye Ridge (90%) and Wakanda Shield (80%) are likely. Thunder Forge is a coin-flip (65%). Captain Liberty is very unlikely (8%).
* **No critical severity (90%):** Hawkeye Ridge and Wakanda Shield are very likely (90%+). Thunder Forge is likely (87%). Black Widow Bay is unlikely (31%).
* **No high or critical (70%):** Hawkeye Ridge (92%) and Wakanda Shield (80%) are likely. Captain Liberty and Iron Sentinel are very unlikely (under 10%).

*Edit the minutes, hours per FTE and targets in Step 2, then choose Run all to see how the answers change.*